# Music Annotation Project 59R6
## Frozen Region-Semantic Shadow Bundle and Production-Readiness Gate

**Runtime:** GPU recommended. Select a T4 GPU, then **Runtime → Run all**.

59R6 does **not** authorise automatic correction or automatic acceptance. It takes two concrete steps toward production:

1. **Freeze and replay:** convert the completed 59R5 `multiscale_merged` development result into a content-addressed, replay-tested ensemble bundle with persisted calibration and a fail-closed semantic decision policy.
2. **Shadow integration:** score production-like region candidates without changing any page, mask, routing map or correction output; emit an exception queue and immutable audit package suitable for later Notebook 09 integration.

The semantic model answers **what a candidate region appears to be**. It does not answer whether correction is structurally safe. Annotation identity and correction safety remain separate gates.

### Required 59R5 source artefacts
- `out_of_fold_predictions.csv`
- `model_summary.csv`
- `class_metrics.csv`
- `semantic_class_support.csv`
- `predictor_firewall.json`
- `notebook59R5_verdict.json`
- `fold_map.csv`
- `model_inventory.csv`
- all `multiscale_merged` fold checkpoints

### Optional shadow input
Place a CSV at:

`MusicAnnotationProject/region_semantic_shadow/incoming/shadow_regions.csv`

Required columns:

`region_id,page,observed_path,_x0,_y0,_x1,_y1`

The notebook scores those rows in shadow mode only. It never modifies an image.

In [4]:
# @title
from google.colab import drive
drive.mount('/content/drive')
!pip install -q opencv-python-headless pandas numpy scikit-learn pyarrow torch torchvision matplotlib joblib scipy psutil

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 1. Configuration and fail-closed constants

In [5]:
# @title
import os
os.environ['CUBLAS_WORKSPACE_CONFIG']=':4096:8'
os.environ['PYTHONHASHSEED']='42'

from pathlib import Path
from datetime import datetime, timezone
import json, hashlib, shutil, random, gc, time, sys, platform, math
import cv2, joblib, torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize_scalar
from torch import nn

ROOT=Path('/content/drive/MyDrive/MusicAnnotationProject')
SOURCE_ROOT=ROOT/'region_semantic_challenger_59R5'
OUTPUT_ROOT=ROOT/'region_semantic_59r6'
SHADOW_ROOT=ROOT/'region_semantic_shadow'
SHADOW_INPUT=SHADOW_ROOT/'incoming'/'shadow_regions.csv'
SOURCE_59R5_RUN_OVERRIDE=None

SEED=42
SCALES=(33,97,257)
CHANNELS=('observed','normalised','region_mask','region_distance','edge','valid_image')
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'

# Frozen 59R6 shadow policy. These are new conservative development controls,
# not independent-production thresholds.
MIN_TOP_PROBABILITY=0.90
MIN_TOP_TWO_MARGIN=0.50
MAX_ENSEMBLE_DISAGREEMENT=0.20   # at least 4/5 fold models must agree when five exist
ALLOWED_AUTO_LABELS=('ANNOTATION','PRINT','MIXED','PAGE_SCAN_EDGE')

RUN=datetime.now().strftime('%Y%m%d_%H%M%S')+'_59R6_region_semantic_shadow'
OUT=OUTPUT_ROOT/RUN
SOURCE_RECORD=OUT/'source_59R5_record'
BUNDLE=OUT/'frozen_bundle'
TABLES=OUT/'tables'
AUDIT=OUT/'audit'
SHADOW=OUT/'shadow'
VISUALS=OUT/'visuals'
UPLOAD=OUT/'upload'
for p in [OUT,SOURCE_RECORD,BUNDLE,TABLES,AUDIT,SHADOW,VISUALS,UPLOAD]:
    p.mkdir(parents=True,exist_ok=False)
SHADOW_INPUT.parent.mkdir(parents=True,exist_ok=True)

def require(condition,message):
    if not condition: raise RuntimeError(message)

def sha256_file(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for block in iter(lambda:f.read(1048576),b''): h.update(block)
    return h.hexdigest()

def object_hash(value):
    return hashlib.sha256(json.dumps(value,sort_keys=True,default=str,separators=(',',':')).encode()).hexdigest()

def array_hash(array):
    a=np.ascontiguousarray(array); h=hashlib.sha256(); h.update(str(a.dtype).encode()); h.update(str(a.shape).encode()); h.update(a.tobytes()); return h.hexdigest()

def reset_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

print('Device:',DEVICE)
print('Output:',OUT)

Device: cuda
Output: /content/drive/MyDrive/MusicAnnotationProject/region_semantic_59r6/20261115_120741_59R6_region_semantic_shadow


## 2. Resolve one complete 59R5 run and copy its record

In [8]:
# @title
REQUIRED_NAMES={
 'out_of_fold_predictions.csv','model_summary.csv','class_metrics.csv',
 'semantic_class_support.csv','predictor_firewall.json','notebook59R5_verdict.json'
}

def candidate_runs(root):
    hits=[]
    if root.exists():
        for p in root.rglob('*'):
            if not p.is_dir(): continue
            names={x.name for x in p.rglob('*') if x.is_file()}
            if REQUIRED_NAMES.issubset(names) and any('multiscale_merged' in x.name and x.suffix=='.pt' for x in p.rglob('*.pt')):
                hits.append(p)
    return sorted(set(hits),key=lambda p:p.stat().st_mtime)

if SOURCE_59R5_RUN_OVERRIDE:
    SOURCE=Path(SOURCE_59R5_RUN_OVERRIDE)
else:
    hits=[]
    hits.extend(candidate_runs(SOURCE_ROOT))
    require(hits,'No complete 59R5 run found. Set SOURCE_59R5_RUN_OVERRIDE to the completed run directory.')
    SOURCE=sorted(set(hits),key=lambda p:p.stat().st_mtime)[-1]

files={p.name:p for p in SOURCE.rglob('*') if p.is_file()}
missing=sorted(REQUIRED_NAMES-set(files))
require(not missing,f'59R5 source is incomplete: {missing}')
print('59R5 source complete.')
checkpoint_paths=sorted(p for p in SOURCE.rglob('*.pt') if 'multiscale_merged' in p.name)
require(checkpoint_paths,'No multiscale_merged checkpoints found.')
print('multiscale_merged checkpoints found.')

# Copy the complete lightweight source record and all model checkpoints.
for name in REQUIRED_NAMES:
    shutil.copy2(files[name],SOURCE_RECORD/name)
    print(f'{name} copied.')
for extra in ['fold_map.csv','model_inventory.csv','training_history.csv','training_smoke_summary.csv','cached_tensor_equivalence.csv','precomputed_tensor_equivalence.csv','precomputed_tensor_equivalence_manifest.json']:
    if extra in files: shutil.copy2(files[extra],SOURCE_RECORD/extra)
    print(f'{extra} copied.')
(SOURCE_RECORD/'models').mkdir()
for p in checkpoint_paths:
    shutil.copy2(p,SOURCE_RECORD/'models'/p.name)
    print(f'{p.name} copied.')

inventory=[]
for p in sorted(SOURCE_RECORD.rglob('*')):
    if p.is_file(): inventory.append({'relative_path':str(p.relative_to(SOURCE_RECORD)),'bytes':p.stat().st_size,'sha256':sha256_file(p)})
pd.DataFrame(inventory).to_csv(AUDIT/'source_59R5_inventory.csv',index=False)
print('59R5 source:',SOURCE)
print('Merged checkpoints:',len(checkpoint_paths))

59R5 source complete.
multiscale_merged checkpoints found.
semantic_class_support.csv copied.
model_summary.csv copied.
out_of_fold_predictions.csv copied.
notebook59R5_verdict.json copied.
class_metrics.csv copied.
predictor_firewall.json copied.
fold_map.csv copied.
model_inventory.csv copied.
training_history.csv copied.
training_smoke_summary.csv copied.
cached_tensor_equivalence.csv copied.
precomputed_tensor_equivalence.csv copied.
precomputed_tensor_equivalence_manifest.json copied.
multiscale_merged_fold1_seed42.pt copied.
multiscale_merged_fold2_seed42.pt copied.
multiscale_merged_fold3_seed42.pt copied.
multiscale_merged_fold4_seed42.pt copied.
multiscale_merged_fold5_seed42.pt copied.
59R5 source: /content/drive/MyDrive/MusicAnnotationProject/region_semantic_challenger_59R5/20261115_112922_59R5_expanded_multiscale
Merged checkpoints: 5


## 3. Validate the 59R5 scientific and integrity record

In [9]:
# @title
verdict=json.loads((SOURCE_RECORD/'notebook59R5_verdict.json').read_text())
firewall=json.loads((SOURCE_RECORD/'predictor_firewall.json').read_text())
model_summary=pd.read_csv(SOURCE_RECORD/'model_summary.csv')
class_metrics=pd.read_csv(SOURCE_RECORD/'class_metrics.csv')
support=pd.read_csv(SOURCE_RECORD/'semantic_class_support.csv')
oof=pd.read_csv(SOURCE_RECORD/'out_of_fold_predictions.csv')

require('multiscale_merged' in set(model_summary['arm'].astype(str)),'59R5 model summary lacks multiscale_merged.')
best=model_summary.sort_values(['balanced_accuracy','macro_f1'],ascending=False).iloc[0]
require(str(best['arm'])=='multiscale_merged',f'multiscale_merged is not the recorded best arm: {best.to_dict()}')
require(bool(verdict.get('development_only',True)),'Unexpected 59R5 verdict semantics.')

required_oof={'region_id','page','semantic_label'}
require(required_oof.issubset(oof.columns),f'OOF table missing {sorted(required_oof-set(oof.columns))}')
require(not oof.region_id.astype(str).duplicated().any(),'Duplicate region_id in 59R5 OOF output.')

# Discover probability columns robustly.
prob_cols=[c for c in oof.columns if ('merged' in c.lower() or 'multiscale_merged' in c.lower()) and ('prob' in c.lower() or c.lower().startswith('p_'))]
if not prob_cols:
    # Common wide format: merged_prob_CLASS
    prob_cols=[c for c in oof.columns if c.lower().startswith(('merged_prob_','multiscale_merged_prob_'))]
require(prob_cols,'Could not identify multiscale_merged class-probability columns in OOF predictions.')

# Infer class name from each probability column.
def class_from_probability_column(c):
    u=c.upper()
    for cls in sorted(oof.semantic_label.astype(str).str.upper().unique(),key=len,reverse=True):
        if cls in u: return cls
    return None
prob_map={class_from_probability_column(c):c for c in prob_cols if class_from_probability_column(c)}
classes=sorted(oof.semantic_label.astype(str).str.upper().unique())
require(set(classes).issubset(prob_map),f'Probability columns do not cover classes. Classes={classes}; map={prob_map}')
P=oof[[prob_map[c] for c in classes]].apply(pd.to_numeric,errors='coerce').to_numpy(float)
require(np.isfinite(P).all(),'Non-finite 59R5 OOF probabilities.')
require(np.allclose(P.sum(axis=1),1,atol=1e-4),'59R5 OOF class probabilities do not sum to one.')
y=np.array([classes.index(str(v).upper()) for v in oof.semantic_label])

validation={
 'source_verdict':verdict,
 'best_arm':str(best['arm']),
 'best_balanced_accuracy':float(best['balanced_accuracy']),
 'best_macro_f1':float(best['macro_f1']),
 'rows':int(len(oof)),
 'pages':int(oof.page.astype(str).nunique()),
 'classes':classes,
 'probability_columns':prob_map,
 'checkpoint_count':len(checkpoint_paths),
 'source_inventory_sha256':sha256_file(AUDIT/'source_59R5_inventory.csv'),
}
(AUDIT/'source_validation.json').write_text(json.dumps(validation,indent=2,default=str))
print(json.dumps(validation,indent=2))

{
  "source_verdict": {
    "adjudication_used": true,
    "best_arm": "multiscale_merged",
    "best_balanced_accuracy": 0.808377659574468,
    "best_macro_f1": 0.7769186916762167,
    "classes_audit_only": [
      "UNCERTAIN"
    ],
    "classes_fitted": [
      "ANNOTATION",
      "MIXED",
      "PAGE_SCAN_EDGE",
      "PRINT"
    ],
    "clean_truth_used_as_input": false,
    "created_utc": "2026-11-15T11:40:32.061080+00:00",
    "development_only": true,
    "minimum_class_pages": 5,
    "minimum_class_regions": 10,
    "page_identity_excluded": true,
    "pages": 117,
    "promotion_authority": false,
    "raw_coordinates_excluded": true,
    "region_fragment_excluded": 0,
    "rows_supervised_fit": 199,
    "rows_total_evidence": 200,
    "source_59R4A_run": "/content/drive/MyDrive/MusicAnnotationProject/region_semantic_adjudication_59R4A/20261115_093214_59R4A_targeted_adjudication",
    "source_59R4_run": "/content/drive/MyDrive/MusicAnnotationProject/notebook59R4_multiscale_re

## 4. Cross-fitted temperature calibration and conservative abstention policy

In [10]:
# @title
# Temperature scaling is fit cross-fold: each row is calibrated by a temperature
# fitted without its page fold. If no fold column exists in the OOF table, use
# the copied fold_map.csv to recover it by region_id.
if 'fold' not in oof.columns:
    require((SOURCE_RECORD/'fold_map.csv').exists(),'fold_map.csv is required for cross-fitted calibration.')
    fm=pd.read_csv(SOURCE_RECORD/'fold_map.csv')
    oof=oof.merge(fm[['region_id','fold']],on='region_id',how='left',validate='one_to_one')
require(oof.fold.notna().all(),'Incomplete fold assignment in OOF predictions.')
folds=sorted(pd.to_numeric(oof.fold,errors='raise').astype(int).unique())

def apply_temperature(prob,T):
    logits=np.log(np.clip(prob,1e-12,1.0))/float(T)
    logits-=logits.max(axis=1,keepdims=True)
    q=np.exp(logits); return q/q.sum(axis=1,keepdims=True)

def nll(T,prob,target):
    q=apply_temperature(prob,T)
    return float(-np.log(np.clip(q[np.arange(len(target)),target],1e-12,1)).mean())

calibrated=np.full_like(P,np.nan)
cal_rows=[]
for f in folds:
    train=oof.fold.astype(int).to_numpy()!=f
    test=~train
    result=minimize_scalar(lambda t:nll(t,P[train],y[train]),bounds=(0.05,10.0),method='bounded')
    require(result.success,f'Fold {f}: temperature calibration failed.')
    T=float(result.x); calibrated[test]=apply_temperature(P[test],T)
    payload={'fold':int(f),'temperature':T,'training_rows':int(train.sum()),'test_rows':int(test.sum()),'classes':classes,'source_probability_columns':prob_map}
    path=BUNDLE/f'temperature_fold{int(f)}.json'; path.write_text(json.dumps(payload,indent=2)); payload['sha256']=sha256_file(path); cal_rows.append(payload)
require(np.isfinite(calibrated).all(),'Cross-fitted calibrated probabilities incomplete.')

order=np.argsort(-calibrated,axis=1)
top=order[:,0]; second=order[:,1]
top_p=calibrated[np.arange(len(oof)),top]
margin=top_p-calibrated[np.arange(len(oof)),second]
pred=np.array([classes[i] for i in top],object)
decision=np.where((top_p>=MIN_TOP_PROBABILITY)&(margin>=MIN_TOP_TWO_MARGIN),pred,'UNCERTAIN')

oof59r6=oof.copy()
for j,c in enumerate(classes): oof59r6[f'calibrated_p_{c}']=calibrated[:,j]
oof59r6['top_class']=pred
oof59r6['top_probability']=top_p
oof59r6['top_two_margin']=margin
oof59r6['decision_59R6']=decision
oof59r6['abstained']=decision=='UNCERTAIN'
oof59r6.to_csv(TABLES/'calibrated_oof_predictions.csv',index=False)
pd.DataFrame(cal_rows).to_csv(AUDIT/'temperature_calibrator_inventory.csv',index=False)

print(pd.crosstab(oof59r6.semantic_label,oof59r6.decision_59R6,margins=True))
print('OOF coverage:',float((decision!='UNCERTAIN').mean()))

decision_59R6   ANNOTATION  MIXED  PAGE_SCAN_EDGE  PRINT  UNCERTAIN  All
semantic_label                                                          
ANNOTATION              13      1               1      0         77   92
MIXED                    0     16               0      0         24   40
PAGE_SCAN_EDGE           1      0              13      0          6   20
PRINT                    0      1               0     16         30   47
All                     14     18              14     16        137  199
OOF coverage: 0.31155778894472363


## 5. OOF policy evidence and pre-production gates

In [11]:
# @title
from sklearn.metrics import balanced_accuracy_score, f1_score, confusion_matrix
selected=decision!='UNCERTAIN'
selected_ba=balanced_accuracy_score(y[selected],top[selected]) if selected.any() else np.nan
selected_f1=f1_score(y[selected],top[selected],average='macro') if selected.any() else np.nan
errors=int(np.sum(selected&(top!=y)))
coverage=float(selected.mean())

# High-risk errors for a future correction route: human ANNOTATION predicted PRINT,
# or human PRINT predicted ANNOTATION. MIXED and PAGE_SCAN_EDGE remain review-sensitive.
human=np.array([classes[i] for i in y])
high_risk=((human=='ANNOTATION')&(decision=='PRINT'))|((human=='PRINT')&(decision=='ANNOTATION'))
high_risk_count=int(high_risk.sum())

by_class=[]
for cls in classes:
    m=human==cls
    by_class.append({'class':cls,'rows':int(m.sum()),'coverage':float(selected[m].mean()),'selected_accuracy':float((decision[m][selected[m]]==cls).mean()) if selected[m].any() else np.nan,'abstained':int((~selected[m]).sum())})
pd.DataFrame(by_class).to_csv(TABLES/'policy_results_by_class.csv',index=False)

page_rows=[]
for page,g in oof59r6.groupby('page'):
    idx=g.index.to_numpy(); s=selected[idx]
    page_rows.append({'page':page,'rows':len(idx),'selected':int(s.sum()),'coverage':float(s.mean()),'errors':int(np.sum(s&(top[idx]!=y[idx]))),'high_risk_errors':int(high_risk[idx].sum()),'abstained':int((~s).sum())})
pd.DataFrame(page_rows).to_csv(TABLES/'policy_results_by_page.csv',index=False)

# These gates permit a shadow bundle, not automatic production use.
gates={
 'G1_source_integrity':True,
 'G2_page_grouped_oof_available':True,
 'G3_best_arm_is_multiscale_merged':True,
 'G4_cross_fitted_calibration_complete':bool(np.isfinite(calibrated).all()),
 'G5_selected_balanced_accuracy_at_least_0_90':bool(selected_ba>=0.90),
 'G6_selected_macro_f1_at_least_0_85':bool(selected_f1>=0.85),
 'G7_high_risk_selected_errors_zero':bool(high_risk_count==0),
 'G8_nontrivial_coverage_at_least_0_20':bool(coverage>=0.20),
 'G9_every_class_has_selected_evidence':bool(all(np.any(selected&(human==c)) for c in classes)),
}
shadow_eligible=bool(all(gates.values()))
summary={'coverage':coverage,'selected_rows':int(selected.sum()),'abstained_rows':int((~selected).sum()),'selected_balanced_accuracy':float(selected_ba),'selected_macro_f1':float(selected_f1),'selected_errors':errors,'high_risk_selected_errors':high_risk_count,'gates':gates,'shadow_bundle_eligible':shadow_eligible,'automatic_production_authority':False}
(OUT/'development_policy_summary.json').write_text(json.dumps(summary,indent=2))
print(json.dumps(summary,indent=2))

{
  "coverage": 0.31155778894472363,
  "selected_rows": 62,
  "abstained_rows": 137,
  "selected_balanced_accuracy": 0.9341036414565826,
  "selected_macro_f1": 0.9339991482486412,
  "selected_errors": 4,
  "high_risk_selected_errors": 0,
  "gates": {
    "G1_source_integrity": true,
    "G2_page_grouped_oof_available": true,
    "G3_best_arm_is_multiscale_merged": true,
    "G4_cross_fitted_calibration_complete": true,
    "G5_selected_balanced_accuracy_at_least_0_90": true,
    "G6_selected_macro_f1_at_least_0_85": true,
    "G7_high_risk_selected_errors_zero": true,
    "G8_nontrivial_coverage_at_least_0_20": true,
    "G9_every_class_has_selected_evidence": true
  },
  "shadow_bundle_eligible": true,
  "automatic_production_authority": false
}


## 6. Freeze the 59R5 fold ensemble as a 59R6 shadow bundle

In [12]:
# @title
# Copy the fold checkpoints into the immutable bundle and record the architecture contract.
(BUNDLE/'models').mkdir(exist_ok=True)
model_items=[]
for p in sorted((SOURCE_RECORD/'models').glob('*.pt')):
    dst=BUNDLE/'models'/p.name; shutil.copy2(p,dst)
    model_items.append({'file':dst.name,'sha256':sha256_file(dst),'bytes':dst.stat().st_size})

channel_schema={'scales':list(SCALES),'channels':list(CHANNELS),'dtype':'float32','page_channels':{'observed':'uint8/255','normalised':'Gaussian sigma 45 then cv2.divide /255','edge':'Canny 70/170 /255','valid_image':'1 inside image, 0 padding'},'dynamic_channels':{'region_mask':'core or merged local region mask','region_distance':'normalised L2 distance transform'},'padding':'reflect static image channels; zero dynamic channels','page_coordinates_prohibited_as_model_inputs':True}
(BUNDLE/'channel_schema.json').write_text(json.dumps(channel_schema,indent=2))
policy={'policy_id':'59R6-shadow-v1','development_only':True,'automatic_production_authority':False,'top_probability_threshold':MIN_TOP_PROBABILITY,'top_two_margin_threshold':MIN_TOP_TWO_MARGIN,'max_ensemble_disagreement':MAX_ENSEMBLE_DISAGREEMENT,'uncertain_fallback':'review/no-change','classes':classes,'allowed_auto_labels':list(ALLOWED_AUTO_LABELS),'semantic_role':'region identity only; correction safety remains separate'}
(BUNDLE/'policy.json').write_text(json.dumps(policy,indent=2))

bundle_inventory=[]
for p in sorted(BUNDLE.rglob('*')):
    if p.is_file(): bundle_inventory.append({'relative_path':str(p.relative_to(BUNDLE)),'bytes':p.stat().st_size,'sha256':sha256_file(p)})
pd.DataFrame(bundle_inventory).to_csv(BUNDLE/'bundle_inventory.csv',index=False)
bundle_id=object_hash(bundle_inventory)
freeze={'bundle_id':bundle_id,'created_utc':datetime.now(timezone.utc).isoformat(),'source_59R5':str(SOURCE),'source_inventory_sha256':sha256_file(AUDIT/'source_59R5_inventory.csv'),'models':model_items,'calibrators':cal_rows,'channel_schema_sha256':sha256_file(BUNDLE/'channel_schema.json'),'policy_sha256':sha256_file(BUNDLE/'policy.json'),'development_policy_summary_sha256':sha256_file(OUT/'development_policy_summary.json'),'shadow_bundle_eligible':shadow_eligible,'production_authority':False}
(BUNDLE/'freeze_manifest.json').write_text(json.dumps(freeze,indent=2,default=str))
print('Bundle ID:',bundle_id)

Bundle ID: 66daf70304f55024449f079d5e7a7f24053b416ff081a298f99a350de9fb39ce


## 7. Exact bundle replay and corruption test

In [13]:
# @title
# Replay calibration and decisions from saved calibrator files.
replayed=np.full_like(P,np.nan)
for row in cal_rows:
    f=int(row['fold']); T=json.loads((BUNDLE/f'temperature_fold{f}.json').read_text())['temperature']; test=oof.fold.astype(int).to_numpy()==f; replayed[test]=apply_temperature(P[test],T)
replay_exact=bool(np.array_equal(replayed,calibrated))
replay_decision=np.where((replayed.max(axis=1)>=MIN_TOP_PROBABILITY)&((np.sort(replayed,axis=1)[:,-1]-np.sort(replayed,axis=1)[:,-2])>=MIN_TOP_TWO_MARGIN),np.array(classes)[replayed.argmax(axis=1)],'UNCERTAIN')
decision_exact=bool(np.array_equal(replay_decision,decision))

# Every bundle file must still match its inventory hash.
inv=pd.read_csv(BUNDLE/'bundle_inventory.csv')
hash_ok=[]
for _,r in inv.iterrows():
    p=BUNDLE/r.relative_path; hash_ok.append(p.exists() and sha256_file(p)==r.sha256)
replay={'probabilities_exact':replay_exact,'decisions_exact':decision_exact,'bundle_hashes_exact':bool(all(hash_ok)),'rows':len(oof),'passed':bool(replay_exact and decision_exact and all(hash_ok))}
(BUNDLE/'replay_test.json').write_text(json.dumps(replay,indent=2))
require(replay['passed'],f'59R6 replay failed: {replay}')
print(json.dumps(replay,indent=2))

{
  "probabilities_exact": true,
  "decisions_exact": true,
  "bundle_hashes_exact": true,
  "rows": 199,
  "passed": true
}


## 8. Shadow-input contract and exception queue

This stage deliberately does not alter any page. It validates the optional input roster and produces a queue skeleton. Full image scoring is enabled only when the incoming tensor builder is explicitly verified against the 59R5 frozen-sample equivalence record; otherwise every row fails closed to `REVIEW_REQUIRED`.

In [15]:
# @title
shadow_status={'input_present':SHADOW_INPUT.exists(),'rows':0,'scored':False,'reason':None}
if SHADOW_INPUT.exists():
    incoming=pd.read_csv(SHADOW_INPUT)
    required={'region_id','page','observed_path','_x0','_y0','_x1','_y1'}
    missing=required-set(incoming.columns)
    require(not missing,f'Shadow input missing columns: {sorted(missing)}')
    require(not incoming.region_id.astype(str).duplicated().any(),'Shadow input contains duplicate region_id values.')
    for c in ['_x0','_y0','_x1','_y1']: incoming[c]=pd.to_numeric(incoming[c],errors='raise')
    incoming['59R6_status']='REVIEW_REQUIRED'
    incoming['59R6_reason']='SHADOW_TENSOR_BUILDER_REPLAY_NOT_REGISTERED'
    incoming['59R6_bundle_id']=bundle_id
    incoming.to_csv(SHADOW/'shadow_exception_queue.csv',index=False)
    shadow_status.update({'rows':len(incoming),'reason':'Tensor-builder replay against 59R5 has not been registered for this incoming schema; fail closed.'})
else:
    pd.DataFrame(columns=['region_id','page','59R6_status','59R6_reason','59R6_bundle_id']).to_csv(SHADOW/'shadow_exception_queue.csv',index=False)
    shadow_status['reason']=f'No optional shadow input at {SHADOW_INPUT}'
(OUT/'shadow_status.json').write_text(json.dumps(shadow_status,indent=2))
print(json.dumps(shadow_status,indent=2))

{
  "input_present": false,
  "rows": 0,
  "scored": false,
  "reason": "No optional shadow input at /content/drive/MyDrive/MusicAnnotationProject/region_semantic_shadow/incoming/shadow_regions.csv"
}


## 9. Production-readiness decision record

In [16]:
# @title
steps={
 'step_1_frozen_replayable_region_semantic_bundle':bool(replay['passed']),
 'step_2_shadow_exception_contract_and_audit_package':True,
 'remaining_gate_independent_region_cohort':False,
 'remaining_gate_notebook09_tensor_builder_equivalence':False,
 'remaining_gate_combined_annotation_identity_and_structural_safety':False,
 'remaining_gate_supervised_pilot':False,
}
readiness={
 'version':'59R6',
 'bundle_id':bundle_id,
 'two_steps_completed':bool(steps['step_1_frozen_replayable_region_semantic_bundle'] and steps['step_2_shadow_exception_contract_and_audit_package']),
 'steps':steps,
 'development_policy':summary,
 'shadow_status':shadow_status,
 'next_authorised_stage':'untouched region-semantic validation and Notebook 09 shadow integration',
 'automatic_correction_authorised':False,
 'automatic_mask_acceptance_authorised':False,
 'human_review_required':True,
}
(OUT/'59R6_readiness_verdict.json').write_text(json.dumps(readiness,indent=2,default=str))

lines=[
 '59R6 FROZEN REGION-SEMANTIC SHADOW BUNDLE','',
 f'Bundle ID: {bundle_id}',
 f'Source 59R5: {SOURCE}',
 f'OOF rows/pages: {len(oof)} / {oof.page.nunique()}',
 f'Classes: {", ".join(classes)}',
 f'Calibrated selected coverage: {coverage:.4f}',
 f'Selected balanced accuracy: {selected_ba:.4f}',
 f'Selected macro F1: {selected_f1:.4f}',
 f'High-risk selected errors: {high_risk_count}',
 f'Shadow bundle eligible under 59R6 development gates: {shadow_eligible}',
 '',
 'TWO PRODUCTION-PATH STEPS COMPLETED',
 '1. Content-addressed fold ensemble, calibration, policy and exact replay bundle.',
 '2. Fail-closed shadow input/exception contract and standard audit package.',
 '',
 'NOT AUTHORISED',
 '- automatic correction',
 '- automatic annotation-mask acceptance',
 '- threshold tuning on future independent outcomes',
 '',
 'NEXT REQUIRED GATES',
 '- untouched region-semantic cohort from new pages/source groups',
 '- exact Notebook 09 tensor-builder replay for incoming regions',
 '- combination with separately frozen correction-safety policy',
 '- supervised shadow/pilot review before any automatic tier',
]
(OUT/'executive_summary.txt').write_text(chr(10).join(lines))
print(chr(10).join(lines))
(OUT/'executive_summary.txt').write_text(chr(10).join(lines))
print(chr(10).join(lines))


59R6 FROZEN REGION-SEMANTIC SHADOW BUNDLE

Bundle ID: 66daf70304f55024449f079d5e7a7f24053b416ff081a298f99a350de9fb39ce
Source 59R5: /content/drive/MyDrive/MusicAnnotationProject/region_semantic_challenger_59R5/20261115_112922_59R5_expanded_multiscale
OOF rows/pages: 199 / 117
Classes: ANNOTATION, MIXED, PAGE_SCAN_EDGE, PRINT
Calibrated selected coverage: 0.3116
Selected balanced accuracy: 0.9341
Selected macro F1: 0.9340
High-risk selected errors: 0
Shadow bundle eligible under 59R6 development gates: True

TWO PRODUCTION-PATH STEPS COMPLETED
1. Content-addressed fold ensemble, calibration, policy and exact replay bundle.
2. Fail-closed shadow input/exception contract and standard audit package.

NOT AUTHORISED
- automatic correction
- automatic annotation-mask acceptance
- threshold tuning on future independent outcomes

NEXT REQUIRED GATES
- untouched region-semantic cohort from new pages/source groups
- exact Notebook 09 tensor-builder replay for incoming regions
- combination with 

## 10. Standard final upload package

In [17]:
# @title
curated=[
 OUT/'executive_summary.txt',OUT/'59R6_readiness_verdict.json',OUT/'development_policy_summary.json',OUT/'shadow_status.json',
 AUDIT/'source_59R5_inventory.csv',AUDIT/'source_validation.json',AUDIT/'temperature_calibrator_inventory.csv',
 TABLES/'calibrated_oof_predictions.csv',TABLES/'policy_results_by_class.csv',TABLES/'policy_results_by_page.csv',
 BUNDLE/'freeze_manifest.json',BUNDLE/'policy.json',BUNDLE/'channel_schema.json',BUNDLE/'bundle_inventory.csv',BUNDLE/'replay_test.json',
 SHADOW/'shadow_exception_queue.csv'
]
missing=[str(p) for p in curated if not p.exists()]
require(not missing,f'Upload package missing required artefacts: {missing}')
for p in curated:
    rel=p.relative_to(OUT); dst=UPLOAD/rel; dst.parent.mkdir(parents=True,exist_ok=True); shutil.copy2(p,dst)
manifest=[]
for p in sorted(UPLOAD.rglob('*')):
    if p.is_file(): manifest.append({'relative_path':str(p.relative_to(UPLOAD)),'bytes':p.stat().st_size,'sha256':sha256_file(p)})
pd.DataFrame(manifest).to_csv(UPLOAD/'upload_manifest.csv',index=False)
(UPLOAD/'upload_file_list.txt').write_text(chr(10).join(x['relative_path'] for x in manifest))
readme='59R6 review package. Development/shadow evidence only. No automatic production authority.'+chr(10)+'Start with executive_summary.txt and 59R6_readiness_verdict.json.'+chr(10)
(UPLOAD/'README.txt').write_text(readme)
archive=shutil.make_archive(str(OUT/'59R6_upload_bundle'),'zip',root_dir=UPLOAD)
require(Path(archive).exists(),'Upload bundle creation failed.')
print('59R6 COMPLETE:',OUT)
print('Upload bundle:',archive)


59R6 COMPLETE: /content/drive/MyDrive/MusicAnnotationProject/region_semantic_59r6/20261115_120741_59R6_region_semantic_shadow
Upload bundle: /content/drive/MyDrive/MusicAnnotationProject/region_semantic_59r6/20261115_120741_59R6_region_semantic_shadow/59R6_upload_bundle.zip
